Example of using Earth Engine Feature Collection data sources

# Ecosystems Map

In [1]:
import logging
logging.getLogger('rle_python_gee.aoo').setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO)

In [2]:
TEST_EXPORTS = True

In [3]:
import ee
from lonboard import Map
import google.auth

from rle_python_gee.ecosystems import Ecosystems
from rle_python_gee.aoo import make_aoo
from rle_python_gee.aoo import wait_for_task

In [4]:
# ee.Initialize(project='goog-rle-assessments')
credentials, _ = google.auth.default(
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/cloud-platform']
)
ee.Initialize(credentials=credentials, project='goog-rle-assessments')

In [5]:
ecosystems = Ecosystems.from_gee_feature_collection(
    'projects/goog-rle-assessments/assets/ruritania/ruritania_ecosystems',
    # 'projects/goog-rle-assessments/assets/colombia/GETCol',
    ecosystem_column='ECO_NAME'
)
ecosystems.load()
ecosystems

EcosystemsEEFeatureCollection(data='projects/goog-rle-assessments/assets/ruritania/ruritania_ecosystems')

In [6]:
ecosystems.to_map()

## AOO

Before running the following command (`make_aoo`):

- Visit the [Code Editor](https://code.earthengine.google.com/)'s Asset tab to verify that the asset folder specified in `gee_asset_path` exists.
- Verify that the GCS storage bucket specified in `gcs_path` exists.

In [7]:
aoo = make_aoo(
    ecosystems,
    gee_asset_path='projects/rle-test-tyler/assets/ruritania/from_fc',
    # gee_asset_path='projects/goog-rle-assessments/assets/colombia'
    gcs_path='gs://rle_test_tyler_bucket/ruritania'
    # gcs_path='gs://rle_test_bucket_1/TEST_columbia'
)
aoo

AOOGridEEFeatureCollection(not computed)

In [8]:
aoo.compute()

INFO:rle_python_gee.aoo:Checking for cached asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid
INFO:rle_python_gee.aoo:Found cached asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid
INFO:gcsfs:gcsfs experimental features enabled via GCSFS_EXPERIMENTAL_ZB_HNS_SUPPORT.
INFO:rle_python_gee.aoo:Loaded grid cells from parquet: gs://rle_test_tyler_bucket/ruritania/aoo_grid.parquet


AOOGridEEFeatureCollection(cell_count=4, aoo_km2=400)

In [9]:
wait_for_task(aoo.task, poll_interval=15)

INFO:rle_python_gee.aoo:No task to wait for — the asset was already cached or compute() has not been called yet.


In [10]:
aoo_polygons = aoo.to_polygons()
aoo_polygons.compute()

INFO:rle_python_gee.aoo:Checking for cached polygons asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid_polygons
INFO:rle_python_gee.aoo:Found cached polygons asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid_polygons
INFO:rle_python_gee.aoo:Loaded polygons from parquet: gs://rle_test_tyler_bucket/ruritania/aoo_grid_polygons.parquet


AOOGridPolygonEEFeatureCollection(polygons=10)

In [11]:
wait_for_task(aoo_polygons.task)

INFO:rle_python_gee.aoo:No task to wait for — the asset was already cached or compute() has not been called yet.


In [12]:
aoo.to_map()

In [13]:
Map(layers=ecosystems.to_layer() + aoo.to_layer())

In [14]:
aoo_grid_polygons = aoo.to_polygons().compute()

INFO:rle_python_gee.aoo:Checking for cached polygons asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid_polygons
INFO:rle_python_gee.aoo:Found cached polygons asset: projects/rle-test-tyler/assets/ruritania/from_fc/aoo_grid_polygons


In [15]:
wait_for_task(aoo.task)

INFO:rle_python_gee.aoo:No task to wait for — the asset was already cached or compute() has not been called yet.


In [16]:
aoo_grid_polygons.to_map()

## Test Exports

## Local exports

In [17]:
if TEST_EXPORTS:
    ecosystems.to_parquet('../tests/test_data/TEST_ecosystems_to_parquet_null_island.parquet')

In [18]:
if TEST_EXPORTS:
    aoo.grid_cells.to_parquet('../tests/test_data/TEST_aoo_grid_cells_to_parquet.parquet')

In [19]:
if TEST_EXPORTS:
    aoo_grid_polygons.polygons.to_parquet('../tests/test_data/TEST_aoo_polygons_to_parquet.parquet')

INFO:rle_python_gee.aoo:Loaded polygons from parquet: gs://rle_test_tyler_bucket/ruritania/aoo_grid_polygons.parquet


## GCS exports

In [20]:
if TEST_EXPORTS:
    ecosystems.to_parquet('gs://rle_test_tyler_bucket/ruritania/from_fc/TEST_ecosystems_to_parquet_null_island.parquet')

In [21]:
if TEST_EXPORTS:
    aoo.grid_cells.to_parquet('gs://rle_test_tyler_bucket/ruritania/from_fc/aoo_grid_cells.parquet')

In [22]:
if TEST_EXPORTS:
    aoo_grid_polygons.polygons.to_parquet('../tests/test_data/TEST_aoo_polygons_to_parquet.parquet')